# Sesión 7 · Seguridad en IA — Tutorial integrador

**Operaciones de Aprendizaje Automático II · CEIA – FIUBA**

Este notebook **integra todo lo visto** y lo mira desde la seguridad. Tomamos un modelo
servido como una API y lo endurecemos **capa por capa**, mapeando cada control al marco
**SAIF** (Secure AI Framework). Después vemos un **ataque adversario** real y su mitigación,
y cerramos con **operación** (auditoría) y **gobernanza** (model card).

### Las capas de seguridad (defensa en profundidad)
| # | Capa | Control de esta sesión | Elemento SAIF |
|---|---|---|---|
| 1 | Autenticación y autorización | API key + JWT | Bases sólidas |
| 2 | Validación y límites | Pydantic + rate limiting | Automatizar defensas |
| 3 | Secretos | variables de entorno / `.env` | Bases sólidas |
| 4 | Datos y modelo | integridad por checksum | Coherencia de controles |
| 5 | Entrenamiento / inferencia robusta | ataque adversario + mitigación | Detectar y actuar |
| — | Operación | logging de auditoría | Detectar y actuar |
| — | Gobernanza | model card | Contextualizar el riesgo / Mejora continua |

> Corre de punta a punta con **uv**, **sin servicios externos**: la API se prueba
> en proceso con `TestClient`. Al final, cada capa tiene su notebook guiado por protocolo
> en `Practica/` (Mini-TP 7).

### Dependencias
```bash
uv add fastapi httpx pyjwt pydantic scikit-learn numpy joblib
# (para usuarios de Poetry: poetry add fastapi httpx pyjwt pydantic scikit-learn numpy joblib)
```


## 0. El modelo que vamos a proteger
Entrenamos un clasificador simple (diagnóstico binario) como *stand-in* del **modelo propio**.
El flujo de seguridad es el mismo con cualquier modelo.


In [ ]:
import numpy as np, joblib, hashlib, json, time, datetime
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

datos = load_breast_cancer()
X, y = datos.data, datos.target
scaler = StandardScaler().fit(X)
Xs = scaler.transform(X)
Xtr, Xte, ytr, yte = train_test_split(Xs, y, test_size=0.25, random_state=42, stratify=y)

modelo = LogisticRegression(max_iter=5000).fit(Xtr, ytr)
print(f"Modelo entrenado. Accuracy de test: {modelo.score(Xte, yte):.3f}")
print(f"Entradas: {X.shape[1]} features | clases: {list(datos.target_names)}")

## Capa 1 — Autenticación y autorización
Nadie debería poder consultar el modelo sin identificarse. Mostramos dos mecanismos:
**API key** (simple) y **JWT** (firmado, con expiración y rol). Son la base de SAIF:
*extender los fundamentos de seguridad al sistema de IA*.


In [ ]:
import jwt   # pyjwt

# En producción, estos secretos vienen de variables de entorno (ver Capa 3), NUNCA del código.
API_KEY = "demo-key-123"
JWT_SECRET = "secreto-de-demo-cambialo-en-produccion-32b+"   # en prod: largo y desde el entorno

def crear_token(usuario: str, rol: str, minutos: int = 15) -> str:
    payload = {"sub": usuario, "rol": rol,
               "exp": datetime.datetime.utcnow() + datetime.timedelta(minutes=minutos)}
    return jwt.encode(payload, JWT_SECRET, algorithm="HS256")

def verificar_token(token: str) -> dict:
    # lanza si está vencido o la firma no coincide
    return jwt.decode(token, JWT_SECRET, algorithms=["HS256"])

tok = crear_token("jaime", "analista")
print("Token JWT emitido:", tok[:40], "...")
print("Verificado ->", verificar_token(tok))

## Capas 1–2 — El servicio endurecido (FastAPI)
Montamos la API con **todas las capas de API juntas**: autenticación, autorización por rol,
**validación de entrada** (Pydantic), **rate limiting** (anti-DoS y anti-scraping) y
**logging de auditoría**. La probamos en proceso con `TestClient` (no hace falta levantar
un servidor).


In [ ]:
from fastapi import FastAPI, Depends, HTTPException, Request
from fastapi.security import APIKeyHeader, HTTPBearer, HTTPAuthorizationCredentials
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field, conlist

app = FastAPI(title="Servicio de modelo (endurecido)")

# ---- Registro de auditoría (Operación) ----
AUDITORIA = []
def auditar(usuario, accion, detalle):
    AUDITORIA.append({"ts": datetime.datetime.utcnow().isoformat(),
                      "usuario": usuario, "accion": accion, "detalle": detalle})

# ---- Capa 1: autenticación (API key) ----
api_key_header = APIKeyHeader(name="X-API-Key", auto_error=False)
def auth_api_key(k: str = Depends(api_key_header)):
    if k != API_KEY:
        auditar("desconocido", "auth_fallida", "api key inválida")
        raise HTTPException(status_code=401, detail="no autorizado")
    return "cliente-apikey"

# ---- Capa 1: autorización por rol (JWT) ----
bearer = HTTPBearer(auto_error=False)
def requiere_rol(rol_minimo="analista"):
    def dep(cred: HTTPAuthorizationCredentials = Depends(bearer)):
        if cred is None:
            raise HTTPException(401, "falta token")
        try:
            datos_tok = verificar_token(cred.credentials)
        except jwt.PyJWTError:
            raise HTTPException(401, "token inválido o vencido")
        if datos_tok.get("rol") not in ("analista", "admin"):
            raise HTTPException(403, "rol insuficiente")
        return datos_tok["sub"]
    return dep

# ---- Capa 2: rate limiting en memoria (anti-DoS / anti-scraping) ----
_ventana = {}
def rate_limit(req: Request, cliente: str = Depends(auth_api_key)):
    ahora = time.time()
    hits = _ventana.setdefault(cliente, [])
    hits[:] = [t for t in hits if ahora - t < 60]      # ventana de 60 s
    if len(hits) >= 5:                                  # máx 5 req/min
        auditar(cliente, "rate_limit", f"{len(hits)} en 60s")
        raise HTTPException(status_code=429, detail="demasiadas solicitudes")
    hits.append(ahora)
    return cliente

# ---- Capa 2: validación de entrada (tipos, rango, tamaño) ----
N_FEATURES = X.shape[1]
class Entrada(BaseModel):
    # exactamente N_FEATURES valores numéricos acotados: frena entradas basura e inyección
    features: conlist(float, min_length=N_FEATURES, max_length=N_FEATURES)

@app.post("/predict")
def predict(x: Entrada, cliente: str = Depends(rate_limit)):
    arr = np.array(x.features, dtype=float).reshape(1, -1)
    # Capa 2: validación de salida -> solo devolvemos la clase y una confianza acotada
    pred = int(modelo.predict(arr)[0])
    conf = float(modelo.predict_proba(arr).max())
    auditar(cliente, "predict", {"hash_entrada": hashlib.sha256(arr.tobytes()).hexdigest()[:12],
                                 "pred": pred})
    return {"prediccion": pred, "confianza": round(conf, 3)}

print("Servicio definido con capas 1–2 + auditoría.")

In [ ]:
cli = TestClient(app)
muestra = {"features": Xte[0].tolist()}

# 1) sin credenciales -> 401
r1 = cli.post("/predict", json=muestra)
# 2) con API key -> 200
h = {"X-API-Key": API_KEY}
r2 = cli.post("/predict", json=muestra, headers=h)
# 3) entrada inválida (longitud incorrecta) -> 422
r3 = cli.post("/predict", json={"features": [0.1, 0.2]}, headers=h)
# 4) rate limiting: a la 6ta en el minuto -> 429
codigos = [cli.post("/predict", json=muestra, headers=h).status_code for _ in range(7)]

print("sin credenciales :", r1.status_code, "(esperado 401)")
print("con API key      :", r2.status_code, r2.json() if r2.status_code==200 else "", "(esperado 200)")
print("entrada inválida :", r3.status_code, "(esperado 422)")
print("rate limiting    :", codigos, "(aparece 429 al superar 5/min)")

**Qué acabamos de frenar**, en términos SAIF:
- **401 sin credenciales** → acceso no autorizado al modelo (bases sólidas).
- **422 entrada inválida** → inyección / entradas basura (automatizar defensas).
- **429 rate limiting** → DoS y **scraping** para robar el modelo por extracción.


## Capa 3 — Secretos y configuración
Las claves (`API_KEY`, `JWT_SECRET`, credenciales del data lake) **nunca van en el código
ni en el repo**. Van en variables de entorno / un archivo `.env` que está en `.gitignore`;
se versiona solo un `.env.example` sin valores reales.


In [ ]:
import os
# Patrón correcto: leer de entorno con un default solo para desarrollo local.
API_KEY = os.environ.get("MODELO_API_KEY", API_KEY)
JWT_SECRET = os.environ.get("MODELO_JWT_SECRET", JWT_SECRET)

ejemplo_env = '''# .env.example  (versionar este; el .env real va en .gitignore)
MODELO_API_KEY=cambiar-en-produccion
MODELO_JWT_SECRET=cambiar-en-produccion
MINIO_ACCESS_KEY=...
MINIO_SECRET_KEY=...'''
print(ejemplo_env)
print("\\nRegla: rotación periódica de claves y mínimo privilegio por servicio.")

## Capa 4 — Integridad del modelo (checksum)
Un atacante que manipule el artefacto del modelo (supply chain) cambia el comportamiento sin
tocar el código. Defensa: calcular un **hash** del artefacto al publicarlo y **verificarlo al
cargarlo**. Si no coincide, no se sirve.


In [ ]:
# Publicar el modelo con su checksum (esto iría al data lake: models/v1/ + models/v1/.sha256)
import io
buf = io.BytesIO(); joblib.dump(modelo, buf); modelo_bytes = buf.getvalue()
sha_publicado = hashlib.sha256(modelo_bytes).hexdigest()
print("SHA-256 publicado:", sha_publicado[:24], "...")

def cargar_verificando(model_bytes, sha_esperado):
    sha = hashlib.sha256(model_bytes).hexdigest()
    if sha != sha_esperado:
        raise ValueError("INTEGRIDAD ROTA: el artefacto fue manipulado; no se carga.")
    return joblib.load(io.BytesIO(model_bytes))

# Carga legítima: pasa
m_ok = cargar_verificando(modelo_bytes, sha_publicado)
print("Carga legítima: OK")

# Carga de un artefacto manipulado: se detecta y se rechaza
manipulado = modelo_bytes[:-1] + bytes([modelo_bytes[-1] ^ 0x01])
try:
    cargar_verificando(manipulado, sha_publicado)
except ValueError as e:
    print("Manipulación detectada ->", e)

## Capa 5 — Ataque adversario y su mitigación
Un **ataque de evasión** perturba la entrada de forma pequeña y deliberada para que el modelo
se equivoque. Lo demostramos con un paso tipo **FGSM** sobre la regresión logística: movemos la
entrada en la dirección que reduce la confianza de la clase correcta.


In [ ]:
w = modelo.coef_[0]

# Tomamos una muestra bien clasificada
idx = next(i for i in range(len(Xte)) if modelo.predict(Xte[i:i+1])[0] == yte[i])
x = Xte[idx].copy()
pred_orig = int(modelo.predict(x.reshape(1, -1))[0])

# FGSM: para flipear, movemos en contra del logit de la clase predicha.
sentido = 1 if pred_orig == 0 else -1
flip_eps = None
for eps in [0.1, 0.2, 0.3, 0.5, 0.8, 1.0]:
    x_adv = x + sentido * eps * np.sign(w)
    if int(modelo.predict(x_adv.reshape(1, -1))[0]) != pred_orig:
        flip_eps = eps; break

print(f"Predicción original: clase {pred_orig} (correcta)")
print(f"Con perturbación ε={flip_eps} (|Δ| medio = {np.abs(x_adv-x).mean():.2f} desvíos estándar):")
print(f"  -> el modelo ahora predice clase {int(modelo.predict(x_adv.reshape(1,-1))[0])}  (¡evasión!)")
print("La entrada cambió muy poco en cada feature, pero la decisión se invirtió.")

In [ ]:
# Generador de ejemplos adversarios (FGSM) para un conjunto completo:
# para cada muestra, movemos en contra del logit de la clase que el modelo predice.
def generar_adversarios(Xset, model, eps):
    p = model.predict(Xset)
    sentido = (1 - 2 * p).reshape(-1, 1)     # +1 si predice 0, -1 si predice 1
    return Xset + sentido * eps * np.sign(model.coef_[0])

EPS = 0.3

# Mitigación 1 (robusta): adversarial training -> reentrenar incluyendo ejemplos perturbados.
X_adv_train = generar_adversarios(Xtr, modelo, EPS)
X_rob = np.vstack([Xtr, X_adv_train]); y_rob = np.concatenate([ytr, ytr])
modelo_robusto = LogisticRegression(max_iter=5000).fit(X_rob, y_rob)

# Evaluamos sobre TODO el test, limpio y atacado:
X_adv_test = generar_adversarios(Xte, modelo, EPS)
print("                 limpio   atacado")
print(f"modelo base      {modelo.score(Xte, yte):.3f}    {modelo.score(X_adv_test, yte):.3f}   <- colapsa bajo ataque")
print(f"modelo robusto   {modelo_robusto.score(Xte, yte):.3f}    {modelo_robusto.score(X_adv_test, yte):.3f}   <- resiste (pequeño costo en limpio)")

# Mitigación 2 (barata): detección de entradas fuera de distribución (OOD).
umbral = 4.0   # desvíos estándar (datos estandarizados: media 0, std 1)
def entrada_sospechosa(arr):
    return bool(np.any(np.abs(arr) > umbral))
print("\\nDefensa en profundidad: rate limiting (encarece el ataque) + OOD + adversarial training.")
print("Ninguna sola basta; juntas, elevan mucho el costo de evadir el modelo.")

## Operación — logging de auditoría
Todo acceso y predicción quedó registrado en `AUDITORIA`. Ese registro es lo que permite
**detectar y actuar** (SAIF): un pico de 429, muchos `auth_fallida` o entradas raras son señales.


In [ ]:
print(f"Eventos de auditoría registrados: {len(AUDITORIA)}")
for e in AUDITORIA[:6]:
    print(" ", e["ts"][11:19], e["usuario"], "·", e["accion"], "·", e["detalle"])
# Conteo por tipo de acción (lo que alimentaría una alerta)
from collections import Counter
print("\\nResumen:", dict(Counter(e["accion"] for e in AUDITORIA)))

## Gobernanza — model card
Una **model card** documenta qué hace el modelo, con qué datos, sus límites y riesgos. Es la
medida de gobernanza concreta que pide el Mini-TP (junto con el audit log).


In [ ]:
model_card = {
    "nombre": "clasificador-diagnostico-v1",
    "tarea": "clasificación binaria (benigno/maligno)",
    "datos": "sklearn breast_cancer (stand-in del modelo propio)",
    "metrica_test": round(float(modelo.score(Xte, yte)), 3),
    "entradas": f"{X.shape[1]} features numéricas estandarizadas",
    "limites": "no validado clínicamente; sensible a entradas fuera de distribución",
    "riesgos": ["evasión adversaria", "fuga de privacidad por inversión"],
    "mitigaciones": ["auth + rate limiting", "validación de entrada", "checksum de integridad",
                     "adversarial training", "logging de auditoría"],
    "responsable": "equipo MLOps II",
    "fecha": datetime.date.today().isoformat(),
}
print(json.dumps(model_card, indent=2, ensure_ascii=False))

## Cierre — SAIF bajado a cada capa de la materia

Lo que hicimos aquí (sobre REST/FastAPI) se replica en cada protocolo. **Ese es el Mini-TP 7**:
hay un notebook guiado por capa y el estudiante elige uno.

| Capa | Riesgo SAIF clave | Control (notebook del Mini-TP) |
|---|---|---|
| **REST** | Abuso, inyección, DoS, scraping | API key/JWT + Pydantic + rate limiting → `mini_tp7_rest_actividad.ipynb` |
| **GraphQL** | DoS por consultas, fuga por introspección | límite de profundidad/costo + authz + sin introspección → `mini_tp7_graphql_actividad.ipynb` |
| **gRPC** | Acceso no autenticado, MITM | interceptor de token + TLS + límite de tamaño → `mini_tp7_grpc_actividad.ipynb` |
| **Streaming** | Envenenamiento, productores no autorizados | validación de esquema + cuarentena (DLQ) → `mini_tp7_streaming_actividad.ipynb` |
| **Federado** | Clientes maliciosos, fuga por updates | agregación robusta + clipping + DP → `mini_tp7_federado_actividad.ipynb` |
| **Data Lake** | Exfiltración, acceso excesivo | acceso mínimo + checksum + URLs prefirmadas → `mini_tp7_datalake_actividad.ipynb` |

➡️ **Mini-TP 7:** elegir una capa, abrir su notebook guiado, aplicar las estrategias SAIF sobre
el modelo propio y agregar una medida de gobernanza (model card o audit log).
